# Human Review: Expert Feedback for GenAI Quality

App developers are most often not domain experts in the business use case, and therefore, need help from domain experts to understand what is a low vs high quality response. MLflow's labeling capabilities enable domain experts to systematically review and label GenAI application traces with ground truth, providing invaluable insights to tune your automated quality metrics and understand how your application should respond.

Key benefits:

- **Expert validation** - Domain experts can review traces and provide structured feedback on quality dimensions
- **Systematic labeling** - Create consistent labeling schemas that capture business-critical quality aspects
- **Quality improvement loop** - Convert expert feedback into training data for LLM judges and evaluation benchmarks

MLflow's Review App provides a pre-built UI, designed for business users, that anyone in your company can use, even if they don't have access to your Databricks workspace.

![human-feedback-overview](https://i.imgur.com/7LNlgDP.gif)

This notebook demonstrates how to collect expert feedback through MLflow's labeling capabilities for the banking customer support assistant. You'll learn to create labeling sessions, add traces for review, and access the Review App for systematic quality improvement through human-in-the-loop evaluation.


## Install packages (only required if running in a Databricks Notebook)

In [ ]:
%pip install -U -r ../requirements.txt
dbutils.library.restartPython()

## Environment Setup

Load environment variables and verify MLflow configuration.


In [ ]:
import sys
sys.path.append('../')
sys.path.append('../../')

import os
import mlflow
from mlflow_demo.utils import *

# Configure for the current participant: derives the object prefix, prompt name,
# model service, and experiment from YOUR username. The widgets only carry the
# shared catalog/schema/warehouse.
setup_databricks_notebook_env()

print('=== Environment Setup ===')
print(f'DATABRICKS_HOST: {os.getenv("DATABRICKS_HOST")}')
print(f'MLFLOW_EXPERIMENT_ID: {os.getenv("MLFLOW_EXPERIMENT_ID")}')
print(f'LLM_MODEL: {os.getenv("LLM_MODEL")}')
print(f'UC_CATALOG: {os.getenv("UC_CATALOG")}')
print(f'UC_SCHEMA: {os.getenv("UC_SCHEMA")}')
print(f'MLFLOW_TRACING_SQL_WAREHOUSE_ID: {os.getenv("MLFLOW_TRACING_SQL_WAREHOUSE_ID")}')
print('Environment variables loaded successfully!')

import logging
logging.getLogger('urllib3').setLevel(logging.ERROR)
logging.getLogger('mlflow').setLevel(logging.ERROR)

In [ ]:
from mlflow_demo.utils import generate_labeling_schema_link, generate_labeling_session_link

# Step 1: Create Labeling Schemas

Define structured feedback schemas that capture the quality dimensions important to your banking application. These schemas guide reviewers to provide consistent, actionable feedback that can be used to improve automated quality assessment.

**Schema Design Best Practices**

- **Simple Yes/No questions** are easier for reviewers and provide clear signals
- **Detailed instructions** help ensure consistent labeling across reviewers
- **Enable comments** to capture qualitative insights beyond the binary rating
- **Focus on business-critical dimensions** rather than generic quality aspects

**Documentation**

- [Expert Feedback](https://docs.databricks.com/aws/en/mlflow3/genai/human-feedback/expert-feedback/label-existing-traces)
- [Labeling Schemas](https://docs.databricks.com/aws/en/mlflow3/genai/human-feedback/concepts/labeling-schemas)

Run the next cell to create banking quality assessment schemas!


In [ ]:
from mlflow.genai import label_schemas

SCORER_PREFIX = os.environ.get('SCORER_PREFIX', '')
p = f'{SCORER_PREFIX}_' if SCORER_PREFIX else ''

schema_configs = {
  f'{p}accuracy': {
    'title': 'Are all financial figures accurate?',
    'instruction': 'Check that all account numbers, balances, transaction amounts, and loan details match the data returned by tools.',
    'type': 'feedback',
  },
  f'{p}relevance': {
    'title': 'Does the response address the question?',
    'instruction': 'Evaluate if the response directly answers the specific banking question asked.',
    'type': 'feedback',
  },
  f'{p}actionability': {
    'title': 'Are the recommendations actionable?',
    'instruction': 'Check if the response provides specific, implementable guidance for a banking customer.',
    'type': 'feedback',
  },
}

print(f'Creating labeling schemas for banking quality assessment (prefix: "{p}")...\n')

In [ ]:
created_schemas = {}
for schema_name, config in schema_configs.items():
  schema = label_schemas.create_label_schema(
    name=schema_name,
    type=config['type'],
    title=config['title'],
    input=label_schemas.InputCategorical(options=['yes', 'no']),
    instruction=config['instruction'],
    enable_comment=True,
    overwrite=True,
  )
  created_schemas[schema_name] = schema
  print(f'Created schema: {schema_name}')
  print(f'  Title: {config["title"]}')
  print(f'  Instructions: {config["instruction"][:80]}...')
  print()

generate_labeling_schema_link();

# Step 2: Create Labeling Session

Create a labeling session that combines the schemas. A Labeling Session is a special type of MLflow Run that organizes a set of traces for review by specific experts using selected labeling schemas. It acts as a queue for the review process.

**Documentation**

- [Expert Feedback](https://docs.databricks.com/aws/en/mlflow3/genai/human-feedback/expert-feedback/label-existing-traces)
- [Labeling Sessions](https://docs.databricks.com/aws/en/mlflow3/genai/human-feedback/concepts/labeling-sessions)

Run the next cell to create a session


In [ ]:
import mlflow

print('Creating labeling session...\n')

session_name = f'banking quality review'

labeling_session = mlflow.genai.create_labeling_session(
  name=session_name,
  assigned_users=[],
  label_schemas=[f'{p}accuracy', f'{p}relevance', f'{p}actionability'],
)

print(f'Created labeling session: {session_name}')
generate_labeling_session_link(labeling_session.labeling_session_id);

# Step 3: Add Traces to Labeling Session

Now we'll add traces to our labeling session so domain experts can review them. This step demonstrates how to programmatically add specific traces that need expert feedback.

### **IMPORTANT**: In this notebook, we show you how to view this data using the MLflow SDKs. You can also perform these same steps using the MLflow Experiment UI.

![add](https://i.imgur.com/mOrdoF5.gif)

**Documentation**

- [Add Traces to Labeling Session](https://docs.databricks.com/aws/en/mlflow3/genai/human-feedback/expert-feedback/label-existing-traces#step-4-generate-traces-and-add-to-the-labeling-session)

Run the next cell to add traces to the labeling session!


In [ ]:
print('Searching for production traces to add...')
traces = mlflow.search_traces(max_results=12, filter_string='tags.sample_data = "yes"')

labeling_session.add_traces(traces)

print(f'Added {len(traces)} traces to labeling session: {session_name}')
generate_labeling_session_link(labeling_session.labeling_session_id);

# Step 4: Share the Review App with your domain experts and label!

The Review App provides an intuitive UI for domain experts to review and label traces. It's specifically designed for business users who do not have direct access to your Databricks workspace but need to provide quality feedback.

**Documentation**

- [Review App](https://docs.databricks.com/aws/en/mlflow3/genai/human-feedback/concepts/review-app)

Run the next cell to get a link to the Review App - open the app and label!


In [ ]:
print('\nReview App URL (for domain experts):')
print(f'   {labeling_session.url}')
print('   Share this link with reviewers to label traces')
print('\n' + '=' * 60)

# Step 5: Accessing and Using Labeling Results

After experts complete labeling, you can see the results in the MLflow UI or access the results through MLflow SDKs to use them to improve your application in several ways:

## Using Human Feedback Results

- **Understand quality patterns** - Identify common issues and strengths
- **Create training data** - Use labeled examples to train custom LLM judges
- **Build evaluation datasets** - Convert labeled traces into systematic test sets
- **Validate automated metrics** - Check if your automated judges align with human assessment
- **Improve prompts** - Use specific feedback to enhance prompt engineering

Run the next cell to see how to programmatically access labeling results!


In [ ]:
from mlflow_demo.utils.mlflow_helpers import get_mlflow_experiment_id

print('ACCESSING LABELING RESULTS')
print('=' * 60)

print('\nAfter experts complete labeling, you can access results via the UI:\n')
generate_labeling_session_link(labeling_session.labeling_session_id)

print('\nAfter experts complete labeling, you can access results programmatically:\n')

traces = mlflow.search_traces(run_id=labeling_session.mlflow_run_id)
traces

# Step 6: Align the Judge to Human Feedback

Your `accuracy` judge started life with *generic* instructions. Domain experts often judge differently than a generic LLM — e.g. a domain-expert reviewer may insist that every balance or transaction claim cite a specific figure. MLflow can **align** a judge to the labels you just collected: it reads the Review App feedback and rewrites the judge's instructions to better match expert judgment.

The loop is: **Review App labels → `judge.align()` → a sharper judge** that you can then reuse for evaluation (Notebook 3) and production monitoring (Notebook 5).

> Alignment needs at least **10 traces** that carry a human `accuracy` label. In a live workshop those come from your experts via the Review App. For a self-contained run, the next cell **tops up** with simulated expert labels (clearly tagged `source_id="simulated_expert"`) so the step always runs — any real human labels always take precedence.


In [ ]:
from mlflow.entities import AssessmentSource, AssessmentSourceType

ALIGN_JUDGE_NAME = 'accuracy'
MIN_LABELS = 10
TARGET_LABELS = 12


def _has_human_label(trace, name=ALIGN_JUDGE_NAME):
    """True if the trace already carries a human Feedback with the given name."""
    for a in trace.search_assessments(name=name, type='feedback'):
        src = getattr(a, 'source', None)
        if getattr(src, 'source_type', None) in (AssessmentSourceType.HUMAN, 'HUMAN'):
            return True
    return False


def _simulated_label(trace):
    """Simple, explainable 'expert' rule: mark 'no' when the response cites no specific
    dollar figure, else 'yes' -- experts want data-backed, specific answers."""
    import re
    try:
        resp = str(trace.data.response or '')
    except Exception:
        resp = ''
    return 'yes' if re.search(r'\$\s?\d|\d[\d,]*\.\d{2}', resp) else 'no'


def ensure_expert_labels(traces, name=ALIGN_JUDGE_NAME, target=TARGET_LABELS):
    """Return traces carrying a human `name` label, topping up with simulated expert
    labels until there are at least `target`. Real labels take precedence."""
    labeled = [t for t in traces if _has_human_label(t, name)]
    candidates = [t for t in traces if not _has_human_label(t, name)]
    newly = []
    for t in candidates[: max(0, target - len(labeled))]:
        mlflow.log_feedback(
            trace_id=t.info.trace_id,
            name=name,
            value=_simulated_label(t),
            source=AssessmentSource(source_type=AssessmentSourceType.HUMAN,
                                    source_id='simulated_expert'),
        )
        newly.append(t.info.trace_id)
    # Re-fetch the just-labeled traces so the new feedback is attached.
    labeled = labeled + [mlflow.get_trace(tid) for tid in newly]
    return labeled


# Pull labeled traces from the EXPERIMENT (not the labeling-session copies) so the
# seeded feedback and get_trace round-trips use stable experiment trace ids.
pool = mlflow.search_traces(filter_string='tags.sample_data = "yes"',
                            max_results=TARGET_LABELS, return_type='list')
if len(pool) < TARGET_LABELS:
    extra = mlflow.search_traces(filter_string='status = "OK"',
                                 max_results=TARGET_LABELS, return_type='list')
    seen = {t.info.trace_id for t in pool}
    pool += [t for t in extra if t.info.trace_id not in seen]

labeled = ensure_expert_labels(pool)
print(f'{len(labeled)} traces carry a human "{ALIGN_JUDGE_NAME}" label (minimum required: {MIN_LABELS}).')


In [ ]:
from mlflow.genai.judges.optimizers import SIMBAAlignmentOptimizer
from mlflow_demo.evaluation.evaluator import accuracy  # the base (generic) judge

print('Aligning the "accuracy" judge to the human labels (this calls the judge model)...\n')
aligned_accuracy = accuracy.align(labeled, optimizer=SIMBAAlignmentOptimizer())

print('=' * 70)
print('BASE instructions')
print('=' * 70)
print(accuracy.instructions)
print('\n' + '=' * 70)
print('ALIGNED instructions')
print('=' * 70)
print(aligned_accuracy.instructions)


### What just happened

`align()` turned your human labels into a revised judge whose instructions encode what the experts actually cared about. To put the aligned judge to work:

- Paste the aligned instructions into the `accuracy` judge in `mlflow_demo/evaluation/evaluator.py`, **or**
- Register the aligned judge as a monitoring scorer in **Notebook 5 (Production Monitoring)**.

You can align `relevance` and `actionability` the same way using their labeling schemas.


# Exercise 1

Create a new labeling schema: 'security_awareness' — does the response avoid exposing sensitive data unnecessarily?


# Exercise 2

Label 3 traces in the Review App. Then retrieve your labels programmatically and compare them to the LLM judge scores.


# Exercise 3

Where do human labels and LLM judges disagree? Discuss why.


# Summary and Next Steps

Congratulations! You've successfully implemented human review workflows for GenAI quality improvement.

## What You've Accomplished

- **Generated traces** for expert review with banking scenarios
- **Created labeling schemas** that capture business-critical quality aspects for banking
- **Set up labeling sessions** accessible through the Review App
- **Learned the workflow** for collecting structured expert feedback

## Key Takeaways

- **Human expertise is invaluable** for understanding quality in your domain
- **Structured schemas** ensure consistent, actionable feedback
- **The Review App** makes it easy for non-technical users to contribute
- **Labels become training data** for automated quality assessment

**Continue Learning**

- [Human Feedback Collection](https://docs.databricks.com/aws/en/mlflow3/genai/human-feedback/) - Complete guide
- [Building Evaluation Datasets](https://docs.databricks.com/aws/en/mlflow3/genai/eval-monitor/build-eval-dataset) - Use labels for evaluation
